# Task 2: Constant Add (Block)

Bot task id: `02_constant_add_block`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/02_constant_add_block.ipynb)

## Условие

Доработаем решение из первого задания: добавим работу с блоками и маскирование хвоста.

В отличие от Task 1, длина вектора `N` может быть произвольной (не обязательно степень двойки), а размер блока `B0` меньше `N`. Поэтому один блок не покрывает весь вектор, и на последнем блоке нужно использовать маску.

$$z[i] = \text{const\_val} + x[i], \quad i = 0 \dots N-1$$

**Требования**
- одна ось идентификатора программы (pid 0);
- блоки размера `B0` обрабатывают вектор по частям;
- использовать `tl.arange`, `tl.load` и `tl.store` с маской для корректной обработки хвоста;
- решение должно работать на GPU (CUDA).

**Подсказка:** `offs = pid * B0 + tl.arange(0, B0)`; при загрузке и записи используйте `mask = offs < N`.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `02_constant_add_block.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 02_constant_add_block.py
import torch
import triton
import triton.language as tl


@triton.jit
def _add_const_block_kernel(x_ptr, z_ptr, const_val, N: tl.constexpr, B0: tl.constexpr):
    # YOUR CODE HERE
    pass


def add_block_triton(x: torch.Tensor, const_val: int) -> torch.Tensor:
    assert x.is_cuda, 'Тензор должен быть на GPU (CUDA).'
    N = x.numel()
    B0 = 128  # например, фиксированный размер блока
    grid = ((N + B0 - 1) // B0,)
    z = torch.empty_like(x)
    _add_const_block_kernel[grid](x, z, const_val, N, B0)
    return z

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "02_constant_add_block.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

for n in (1, 127, 1000, 4097):
    x = torch.randn(n, device="cuda")
    torch.testing.assert_close(sol.add_block_triton(x, -3), x - 3)

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("02_constant_add_block.py")